[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpszr/litert-llm-cookbook/blob/main/colab/09_streaming_with_system_prompt.ipynb)

# Example 09: Streaming with System Prompt

Combines a system persona with streaming output.

The model is told to act as a senior software engineer specialising in on-device AI.
Tokens are printed to the cell output as they arrive rather than waiting for the full response.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import subprocess
subprocess.run([
    "curl", "-L",
    "https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/main/gemma-4-E2B-it.litertlm?download=true",
    "-o", "/content/gemma-4-E2B-it.litertlm"
], check=True)

In [ ]:
import litert_lm

MODEL_PATH = "/content/gemma-4-E2B-it.litertlm"

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": (
                    "You are a senior software engineer at Google specialising in "
                    "on-device AI and TensorFlow Lite. Give concise, technical answers."
                ),
            }
        ],
    }
]

with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation(messages=messages) as conversation:
        user_input = "What are the main advantages of running LLMs on-device vs in the cloud?"
        print(f"Q: {user_input}\nA: ", end="", flush=True)
        for chunk in conversation.send_message_async(user_input):
            for item in chunk.get("content", []):
                if item.get("type") == "text":
                    print(item["text"], end="", flush=True)
        print()